# PHASE 1–2 — NIFTY IV Data Acquisition & EDA

This notebook is the exploratory/data-understanding layer of the project.

**Official FinanceMeta benchmark:** `notebooks/02_reproducible_benchmark.ipynb`

The benchmark protocol, frozen before final comparative results, is documented in
`docs/protocol.md`. This EDA notebook is not used to tune the final benchmark.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data").exists():
    REPO_ROOT = REPO_ROOT.parent
DATA_PATH = REPO_ROOT / "data" / "dataset.csv"

df = pd.read_csv(DATA_PATH)
df["datetime"] = pd.to_datetime(df["datetime"], format="%d-%m-%Y %H:%M", errors="raise")

option_cols = [
    c for c in df.columns
    if c.startswith("NIFTY")
]

print("Dataset shape:", df.shape)
print("Time range:", df["datetime"].min(), "to", df["datetime"].max())
print("Option columns:", len(option_cols))
print("Natural missing IV rate:",
      df[option_cols].isna().mean().mean())

## Structural characteristics relevant to the benchmark

The dataset should be inspected for:

- timestamp coverage
- strike coverage
- option type coverage
- expiry count
- natural missingness
- IV distribution
- smile shape

A key scope limitation must remain visible: the current repository dataset has
one expiry, so the FinanceMeta checkpoint is a cross-sectional strike/moneyness
reconstruction benchmark rather than a multi-expiry term-structure benchmark.

In [ ]:
# Benchmark-relevant structural summary
metadata_rows = []
for symbol in option_cols:
    expiry_str = symbol[5:12]
    strike_str = symbol[12:-2]
    option_type = symbol[-2:]
    metadata_rows.append({
        "symbol": symbol,
        "expiry": pd.to_datetime(expiry_str, format="%d%b%y"),
        "strike": float(strike_str),
        "option_type": option_type,
    })

meta = pd.DataFrame(metadata_rows)

print("Unique strikes:", meta["strike"].nunique())
print("Unique expiries:", meta["expiry"].nunique())
print("Option types:", sorted(meta["option_type"].unique().tolist()))
print("Timestamp frequency (mode):", df["datetime"].diff().dropna().mode().iloc[0])

## Data provenance and rights

The final submission must use the verified original source and rights wording.
Do not infer redistribution rights from the presence of the CSV in GitHub.

Before submission, fill the README/evidence note with the exact verified
source, acquisition route, and permitted use/redistribution terms.

## Historical benchmark note

The original version of this notebook/project explored missingness and
interpolation. Those exploratory observations remain useful for project
history, but the official FinanceMeta results are generated by the frozen
benchmark in `02_reproducible_benchmark.ipynb`.

The benchmark must be run without using this EDA notebook to tune model
settings on the final evaluation holdout.